## 0. Setup
Runtime → *Change runtime type* → **T4 GPU** (needed for training only; the submission check runs on CPU).

The cell below:
1. mounts Google Drive,
2. installs **exactly** the library versions Codabench uses (FLOP counts depend on the torch version),
3. copies the student package from Drive to the Colab disk (reading 15k images straight from Drive is slow),
4. links `data/checkpoints` back to Drive so checkpoints survive a disconnect.

Put the unzipped student package in `MyDrive/vqa_datathon/` first (or set `PKG_ZIP` to the zip on Drive).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/vqa_datathon'   # unzipped student package lives here
PKG_ZIP = None                                    # or e.g. '/content/drive/MyDrive/vqa_datathon_student_package.zip'
WORK = '/content/vqa_datathon'                    # fast local copy

In [ ]:
%%capture pipout
!pip install torch==2.14.1 torchvision==0.29.1 transformers==5.20.2 pillow==12.3.0 numpy==2.2.6 tqdm==4.70.1 pyyaml==6.0.3

In [ ]:
import os, subprocess, torch, torchvision, transformers
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| transformers', transformers.__version__,
      '| CUDA', torch.cuda.is_available())
assert torch.__version__.startswith('2.14.1'), 'Restart the runtime (Runtime -> Restart session) and rerun from the top'

if not os.path.exists(WORK):
    if PKG_ZIP:
        subprocess.run(['unzip', '-q', PKG_ZIP, '-d', '/content'], check=True)
        os.rename('/content/student_package', WORK)
    else:
        subprocess.run(['cp', '-r', DRIVE, WORK], check=True)
os.makedirs(f'{DRIVE}/checkpoints', exist_ok=True)
os.makedirs(f'{WORK}/data', exist_ok=True)
ck = f'{WORK}/data/checkpoints'
if not os.path.islink(ck):
    subprocess.run(['rm', '-rf', ck]); os.symlink(f'{DRIVE}/checkpoints', ck)
os.chdir(WORK)
print(open('README.md').read()[:600])

## 1. Train the baseline
Frozen ResNet-18 grid features + GRU question encoder + 2-glimpse attention + fusion MLP + 1000-way classifier,
trained with soft VQA targets on **train**, best epoch selected on **val**. About 10 minutes on a T4.
The best checkpoint is saved to `MyDrive/vqa_datathon/checkpoints/simple_baseline.pt` whenever val improves.

Expected: val ≈ 46.2, params 18,154,866, FLOPs 3,688,267,776.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python baseline/train_simple.py

## 2. Evaluate on val with the official-metric evaluator

In [ ]:
!python eval/vqa_eval.py --pred data/preds/simple_val.json --gt data/processed/val/annotations.json

In [ ]:
import json
r = json.load(open('logs/simple_results.json'))
print('complexity:', r['complexity'])
for h in r['history'][::5]: print(h['epoch'], h['loss'], h['val'])

## 3. Your turn
- Edit `baseline/simple/model.py` (or write your own `model.py` with `build_model()` and `predict(images, questions)`).
- Keep the interface: `images` is a FloatTensor[B,3,224,224] in [0,1] (RGB, **not** normalized), `questions` is a list of str,
  and the return value is a list of lowercase answer strings.
- Score = Acc − 2·log2(Params/18.15M) − 2·log2(FLOPs/3.69G): every doubling of params or FLOPs costs 2 points.
- When ready, run `notebooks/make_submission.ipynb`.